# Tahap 3a: Pelatihan LoRA di GPU Server
Subbab 3.6.7 dan Gambar 4.8 bagian pelatihan. Bagian lain Tahap 3 (penggabungan, konversi, pemeriksaan, transkripsi data uji, analisis) dikerjakan di laptop dengan `11b_tahap3_evaluasi.ipynb`, supaya seluruh pengukuran tetap di perangkat Tabel 3.2.

```
[1.1] Uji coba 2 epoch -> data/output/tahap3/uji_coba/ (terpisah, bukan hasil penelitian)
[1.2] Pelatihan LoRA   -> data/output/tahap3/latih/ (konfigurasi, log_langkah, log_validasi, validasi_segmen, sesi, adapter_terbaik/)
[2] Metadata pelatihan -> data/output/tahap3/latih/run_meta_latih.json, kurva_pelatihan.png
```

## 0. Persiapan

In [ ]:
from pathlib import Path
import sys, os, json, math, datetime, importlib, subprocess, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
import src.latih_lora as ll
import src.kurva_pelatihan as kp
import src.transkripsi as tr
for m in (pt, ll, kp, tr):
    importlib.reload(m)

OUT        = ROOT / "data" / "output"
MANIFEST   = OUT / "manifest_rekaman.csv"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
ACUAN_DIR  = OUT / "praproses_teks" / "acuan"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
SEG_LATIH  = OUT / "segmen_latih.csv"
SEG_AUDIO  = ROOT / "data" / "processed" / "segmen_latih"
T1         = OUT / "tahap1"
T2         = OUT / "tahap2"
T3         = OUT / "tahap3"
LATIH      = T3 / "latih"          # hasil pelatihan; satu-satunya folder yang dibawa pulang dari server
KERJA      = T3 / "_kerja"
STATE      = KERJA / "state"       # state lengkap untuk melanjutkan pelatihan (besar, tidak di-commit)
MODEL      = T3 / "model"          # model hasil gabung dan konversi (besar, tidak di-commit)
MENTAH     = T3 / "mentah"
LOG_WAKTU  = T3 / "waktu_transkripsi.jsonl"
for d in (T3, LATIH, KERJA):
    d.mkdir(parents=True, exist_ok=True)

rel = lambda p: Path(p).relative_to(ROOT).as_posix()

In [ ]:
# Varian terpilih dan konfigurasi Tahap 1
pemilihan = json.loads((T1 / "pemilihan_varian.json").read_text(encoding="utf-8"))
meta_t1 = json.loads((T1 / "run_meta_tahap1.json").read_text(encoding="utf-8"))
VARIAN = pemilihan["terpilih"]
KODE_DASAR, KODE_T2, KODE = pemilihan["kode_baseline"], "T2", "T3"

manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False, encoding="utf-8-sig").set_index("file_id")
uji = sorted(manifest.index[manifest.split == "uji"])
validasi = sorted(manifest.index[manifest.split == "validasi"])
assert len(uji) == 9 and len(validasi) == 4, f"{len(uji)} rekaman uji, {len(validasi)} rekaman validasi"

kamus = pt.muat_kamus(KAMUS_PATH)
assert kamus.terkunci and kamus.tanggal_kunci == meta_t1["kamus"]["tanggal_kunci"], "kamus berbeda dengan Tahap 1"
seg = pd.read_csv(SEG_LATIH, dtype={"seg_latih_id": str}, keep_default_na=False, encoding="utf-8-sig")
assert set(seg.loc[seg.split == "validasi", "file_id"]) == set(validasi), "segmen validasi tidak sama dengan manifest"
assert not set(seg.file_id) & set(uji), "ada segmen latih/validasi dari rekaman uji"

print(f"Varian: {VARIAN}, baseline {KODE_DASAR} dan {KODE_T2}")
print(f"Segmen: {seg.split.value_counts().to_dict()}; rekaman validasi: {validasi}")
print(f"Kamus terkunci sejak {kamus.tanggal_kunci}")

In [ ]:
# WAV segmen tidak ada di GitHub, jadi keberadaan dan formatnya diperiksa dulu.
SPLIT_DIPERIKSA = ['latih', 'validasi']
masalah, beda_durasi = [], []
dicek = seg[seg.split.isin(SPLIT_DIPERIKSA)]
for r in dicek.itertuples():
    p = SEG_AUDIO / r.split / f"{r.seg_latih_id}.wav"
    if not p.exists():
        masalah.append((r.seg_latih_id, "tidak ada"))
        continue
    i = sf.info(str(p))
    if (i.samplerate, i.channels) != (16000, 1):
        masalah.append((r.seg_latih_id, f"{i.samplerate} Hz, {i.channels} kanal"))
    elif abs(i.frames / i.samplerate - float(r.dur)) > 0.1:
        beda_durasi.append((r.seg_latih_id, round(i.frames / i.samplerate, 2), float(r.dur)))
masalah = pd.DataFrame(masalah, columns=["seg_latih_id", "masalah"])
print(f"WAV {'/'.join(SPLIT_DIPERIKSA)}: {len(dicek) - len(masalah)}/{len(dicek)} siap (16 kHz mono)")
if beda_durasi:
    print(f"[!] {len(beda_durasi)} WAV berbeda durasi > 0,1 detik dari segmen_latih.csv:", beda_durasi[:5])
assert masalah.empty, "WAV bermasalah:\n" + masalah.head(20).to_string()

In [ ]:
# Mesin dipakai bersama: lihat pemakaian tiap GPU, lalu kunci pelatihan ke satu GPU yang luang.
kolom = ["index", "name", "memory.used", "memory.total", "utilization.gpu"]
out = subprocess.check_output(["nvidia-smi", f"--query-gpu={','.join(kolom)}", "--format=csv,noheader,nounits"])
gpu = pd.DataFrame([[x.strip() for x in b.split(",")] for b in out.decode().strip().splitlines()], columns=kolom)
gpu[["index", "memory.used", "memory.total", "utilization.gpu"]] = gpu[["index", "memory.used", "memory.total",
                                                                      "utilization.gpu"]].astype(int)
print(gpu.to_string(index=False))

GPU_ID = None   # isi nomor GPU bila ingin memilih sendiri; None = GPU dengan memori terpakai paling kecil
if GPU_ID is None:
    GPU_ID = int(gpu.sort_values(["memory.used", "utilization.gpu"]).iloc[0]["index"])
g = gpu.set_index("index").loc[GPU_ID]
GPU = {"index": GPU_ID, "nama": g["name"], "vram_total_mb": int(g["memory.total"]),
       "vram_terpakai_mb": int(g["memory.used"])}
print(f"\nDipakai: GPU {GPU_ID} ({g['name']}), {g['memory.used']} MB terpakai, utilisasi {g['utilization.gpu']}%")
if g["memory.used"] > 1000 or g["utilization.gpu"] > 10:
    print("[!] GPU ini sedang dipakai orang lain. Pilih GPU lain atau tunggu sampai luang.")

## 1. Pelatihan LoRA
Konfigurasi Tabel 3.6. Nilai LoRA dan laju belajar dibaca dari uji kelayakan Tahap 1, ukuran batch dan gradient checkpointing dari `pemilihan_varian.json`, lalu akumulasi gradien mengembalikan batch efektif ke 4.
- Satu langkah optimizer berisi 4 segmen. 1.001 segmen latih tidak habis dibagi 4, jadi langkah terakhir tiap epoch berisi sisa segmen dan loss-nya dirata-rata atas jumlah segmen yang sebenarnya.
- WER validasi (greedy, tanpa timestamp, total kesalahan dibagi total kata acuan 149 segmen) dihitung di langkah 0 sebagai pembanding, tiap 200 langkah, dan di langkah terakhir. Langkah 0 tidak ikut dipilih.
- Konfigurasi dikunci di `latih/konfigurasi.json`. Bila diubah setelah pelatihan berjalan, worker menolak melanjutkan.

In [ ]:
EPOCH, BATCH_EFEKTIF, RASIO_PEMANASAN, EVAL_TIAP, SEED = 35, 4, 0.1, 200, 42
LORA = meta_t1["uji_kelayakan"]["lora"]
assert LORA == {"r": 32, "alpha": 64, "dropout": 0.0, "target_modules": ["q_proj", "v_proj"]}, LORA
BATCH = pemilihan["batch_tahap3"]

CFG_LATIH = {
    "nama": KODE, "model_id": f"openai/whisper-{VARIAN}", "lora": LORA,
    "laju_belajar": meta_t1["uji_kelayakan"]["laju_belajar"], "epoch": EPOCH,
    "batch": BATCH, "akumulasi": BATCH_EFEKTIF // BATCH, "rasio_pemanasan": RASIO_PEMANASAN,
    "gradient_checkpointing": pemilihan["gradient_checkpointing"], "seed": SEED,
    "eval_tiap": EVAL_TIAP, "eval_langkah_akhir": True, "eval_batch": 2, "maks_token_baru": 440,
    # nilai bawaan torch.optim.AdamW, sama dengan uji kelayakan
    "adamw": {"betas": [0.9, 0.999], "eps": 1e-8, "weight_decay": 0.01},
    "segmen_latih": rel(SEG_LATIH), "audio_dir": rel(SEG_AUDIO), "kamus": rel(KAMUS_PATH),
    "glosarium": rel(GLOSARIUM), "folder": rel(LATIH), "folder_state": rel(STATE), "perangkat": "cuda",
}
assert BATCH * CFG_LATIH["akumulasi"] == BATCH_EFEKTIF
R = ll.rencana(int((seg.split == "latih").sum()), BATCH, CFG_LATIH["akumulasi"], EPOCH, RASIO_PEMANASAN)
PATH_CFG = KERJA / "latih_lora.json"

print(pd.Series({"model": CFG_LATIH["model_id"], "LoRA": f"r {LORA['r']}, alpha {LORA['alpha']}, dropout {LORA['dropout']}, "
                 f"{'/'.join(LORA['target_modules'])} di encoder dan decoder",
                 "pengoptimal": f"AdamW, lr {CFG_LATIH['laju_belajar']}, linear decay",
                 "batch": f"{BATCH} x akumulasi {CFG_LATIH['akumulasi']} = {BATCH_EFEKTIF}",
                 "gradient checkpointing": CFG_LATIH["gradient_checkpointing"],
                 "langkah optimizer": f"{R['langkah_per_epoch']} per epoch x {EPOCH} = {R['total_langkah']}",
                 "pemanasan": f"{R['langkah_pemanasan']} langkah",
                 "validasi": f"langkah 0, tiap {EVAL_TIAP}, dan langkah {R['total_langkah']} "
                             f"({R['total_langkah'] // EVAL_TIAP + 1} kali dapat dipilih)",
                 "seed": SEED}).to_string())

### 1.1 Uji coba singkat
Sebelum pelatihan penuh, konfigurasi yang sama dijalankan 2 epoch (502 langkah) ke folder terpisah `tahap3/uji_coba/`, untuk memastikan kode berjalan di server, memperkirakan lama pelatihan penuh, dan memeriksa puncak memori. Uji coba dijalankan dua putaran: putaran pertama berhenti sendiri setelah simpanan pertama (langkah 200), putaran kedua melanjutkannya sampai selesai, sehingga fitur lanjut-setelah-terputus ikut teruji.

Uji coba hanya memeriksa bahwa pelatihan berjalan. Tabel 3.6 tidak diubah berdasarkan hasilnya, dan pelatihan penuh dimulai dari awal dengan state sendiri. Karena jumlah epoch berbeda, jadwal laju belajar uji coba juga lebih pendek.

In [ ]:
JALANKAN_UJI_COBA = True
UJI_EPOCH = 2
UJI = T3 / "uji_coba"
CFG_UJI = CFG_LATIH | {"nama": "T3-uji", "epoch": UJI_EPOCH, "folder": rel(UJI / "latih"),
                       "folder_state": rel(UJI / "_kerja" / "state")}

if JALANKAN_UJI_COBA:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
    print("Putaran 1: berhenti sendiri setelah simpanan pertama\n")
    tr.jalankan_terpisah(CFG_UJI | {"batas_jam": 1e-6}, KERJA / "uji_coba.json", cwd=ROOT, modul="src.latih_lora")
    print("\nPutaran 2: melanjutkan dari state sampai selesai\n")
    tr.jalankan_terpisah(CFG_UJI | {"batas_jam": None}, KERJA / "uji_coba.json", cwd=ROOT, modul="src.latih_lora")
else:
    print("JALANKAN_UJI_COBA False: memakai hasil yang sudah ada di", rel(UJI))

In [ ]:
du = kp.muat(UJI / "latih")
L, V, S = du["langkah"], du["validasi"], du["sesi"]
total_uji = du["konfigurasi"]["rencana"]["total_langkah"]
e = kp.per_epoch(du)
vram = int(V.vram_puncak_dicadangkan_mb.max()) if "vram_puncak_dicadangkan_mb" in V else None
cek = pd.DataFrame([
    ("semua langkah selesai", len(L) == total_uji, f"{len(L)}/{total_uji}"),
    ("dilanjutkan dari state tanpa langkah ganda", len(S) >= 2 and L.langkah.is_unique
        and bool((L.langkah.diff().dropna() == 1).all()), f"{len(S)} sesi"),
    ("loss selalu terhingga", bool(np.isfinite(L.loss).all()), f"{L.loss.min():.3f}-{L.loss.max():.3f}"),
    ("adapter terbaik tersimpan", (UJI / "latih" / "adapter_terbaik" / "adapter_model.safetensors").exists(), ""),
    ("loss validasi tercatat", bool(V.loss_validasi.notna().all()), ""),
    ("langkah dilewati GradScaler sedikit", int(L.dilewati.sum()) <= 5, f"{int(L.dilewati.sum())} langkah"),
    ("loss latih epoch terakhir < epoch 1", bool(e.loss_latih.iloc[-1] < e.loss_latih.iloc[0]),
        " -> ".join(f"{x:.4f}" for x in e.loss_latih)),
    ("memori GPU cukup", vram is None or vram < GPU["vram_total_mb"], f"puncak {vram} MB dari {GPU['vram_total_mb']} MB"),
], columns=["pemeriksaan", "lolos", "keterangan"])
print(cek.to_string(index=False))
WAJIB = cek.pemeriksaan.iloc[:5]
assert cek.set_index("pemeriksaan").loc[WAJIB, "lolos"].all(), "uji coba gagal; kirim isi sel di atas dan konsol.log"

detik = L.detik.median()
n_val = R["total_langkah"] // EVAL_TIAP + 2
jam = (R["total_langkah"] * detik + n_val * V.detik_eval.median()) / 3600
print(f"\n{detik:.2f} detik per langkah optimizer, {V.detik_eval.median():.0f} detik per validasi")
print(f"Perkiraan pelatihan penuh: {R['total_langkah']} langkah + {n_val} validasi = +-{jam:.1f} jam")
t = V[["langkah", "epoch", "WER", "loss_validasi"]].copy()
t["WER"] = (100 * t.WER).round(2)
print(t.round(4).to_string(index=False))
kp.gambar_epoch(du, UJI / "kurva_per_epoch.png")
plt.show()
print("Dengan laju belajar 1e-5, perubahan WER dalam 2 epoch bisa kecil; yang diperiksa di sini arahnya wajar dan tidak meledak.")

### 1.2 Pelatihan penuh
Pelatihan sebaiknya dijalankan dari terminal di dalam `tmux`, supaya tetap berjalan bila VPN atau SSH terputus. Sel berikut menulis konfigurasi dan mencetak perintahnya. `JALANKAN_DARI_NOTEBOOK = True` menjalankannya langsung dari sel, tetapi prosesnya ikut berhenti bila kernel mati.

In [ ]:
JALANKAN_DARI_NOTEBOOK = False
BATAS_JAM = None   # mis. 8: berhenti rapi pada simpanan pertama setelah 8 jam; jalankan ulang untuk melanjutkan

cfg = CFG_LATIH | {"batas_jam": BATAS_JAM}
PATH_CFG.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding="utf-8")
perintah = (f"CUDA_VISIBLE_DEVICES={GPU_ID} python -m src.latih_lora {rel(PATH_CFG)} "
            f"2>&1 | tee -a {rel(LATIH / 'konsol.log')}")
print(f"""Jalankan di terminal server:

  tmux new -s tahap3              (atau: tmux attach -t tahap3 bila sesi sudah ada)
  cd {ROOT.as_posix()}
  source .venv/bin/activate
  {perintah}

Lepas dari tmux tanpa menghentikan pelatihan: Ctrl+B lalu D.
Menjalankan perintah yang sama lagi melanjutkan dari state terakhir.""")

if JALANKAN_DARI_NOTEBOOK:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
    tr.jalankan_terpisah(cfg, PATH_CFG, cwd=ROOT, modul="src.latih_lora")

### 1.3 Pantau pelatihan
Loss per langkah optimizer dan WER validasi per evaluasi, dibaca dari log. Garis putus-putus adalah WER langkah 0 (model dasar dengan transformers). Jalankan ulang sel ini kapan saja selama pelatihan berlangsung.

In [ ]:
d = kp.muat(LATIH)
konf, L, V = d["konfigurasi"], d["langkah"], d["validasi"]
total = konf["rencana"]["total_langkah"]
g = int(L.langkah.max()) if len(L) else 0
print(f"Langkah {g}/{total} ({100 * g / total:.1f}%), epoch {g / d['langkah_per_epoch']:.2f}, {len(d['sesi'])} sesi; "
      f"dilewati GradScaler: {int(L.dilewati.sum()) if len(L) else 0}; "
      f"norma gradien tak hingga/NaN: {int(L.norma_gradien.isna().sum()) if len(L) else 0}")
if len(V):
    kp.gambar_langkah(d, LATIH / "kurva_pelatihan.png")
    plt.show()
    t = V[["langkah", "epoch", "WER", "loss_validasi", "S", "D", "I", "N", "detik_eval", "terbaik"]].copy()
    t["WER"] = (100 * t.WER).round(2)
    print(t.round(3).tail(10).to_string(index=False))

In [ ]:
if len(d["langkah"]):
    kp.gambar_epoch(d, LATIH / "kurva_per_epoch.png")
    plt.show()
    print(kp.ringkas_konvergensi(d).to_string())

## 2. Metadata pelatihan dan pemindahan hasil
Perangkat dan versi pustaka di server dicatat bersama hasil pelatihan, karena pelatihan tidak dijalankan di perangkat Tabel 3.2. Folder `latih/` lalu disalin ke laptop untuk notebook 11b.

In [ ]:
import platform, getpass, importlib.metadata as im
def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return None

meta_latih = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "Tahap 3a: pelatihan LoRA di GPU server",
    "host": platform.node(), "platform": platform.platform(), "python": sys.version.split()[0],
    "gpu": GPU, "gpu_semua": gpu.to_dict("records"),
    "konfigurasi": CFG_LATIH, "rencana": R,
    "sesi": ll.baca_jsonl(LATIH / "sesi.jsonl"),
    "versi_pustaka": {p: versi(p) for p in ["torch", "transformers", "peft", "accelerate", "tokenizers", "jiwer",
                                             "numpy", "pandas", "soundfile"]},
}
(LATIH / "run_meta_latih.json").write_text(json.dumps(meta_latih, ensure_ascii=False, indent=2, default=str),
                                           encoding="utf-8")
beda_versi = {p: (v, meta_t1["versi_pustaka"].get(p)) for p, v in meta_latih["versi_pustaka"].items()
              if p in meta_t1["versi_pustaka"] and v != meta_t1["versi_pustaka"][p]}
print("Versi pustaka berbeda dengan Tahap 1:", beda_versi or "tidak ada")

selesai = len(ll.baca_jsonl(LATIH / "log_langkah.jsonl")) == R["total_langkah"]
ukuran = sum(f.stat().st_size for f in LATIH.rglob("*") if f.is_file()) / 2**20
print(f"""
Pelatihan {'selesai' if selesai else 'BELUM selesai'}. Salin folder latih/ ({ukuran:.0f} MB) ke laptop.
Dari terminal laptop, di folder repo:

  scp -P 1004 -r {getpass.getuser()}@gpu-labkc.ub.ac.id:{LATIH.as_posix()} data/output/tahap3/
""")

## Output

| Berkas di `data/output/tahap3/` | Isi | Dipakai di |
|---|---|---|
| `latih/konfigurasi.json` | Konfigurasi Tabel 3.6 yang dikunci beserta rencana langkah | Bab 6, lampiran |
| `latih/log_langkah.jsonl`, `latih/log_validasi.jsonl`, `latih/kurva_pelatihan.png` | Loss latih per langkah optimizer, WER dan loss validasi per evaluasi, dan kurvanya | Bab 6, pemantauan laju belajar |
| `latih/validasi_segmen.jsonl` | Hipotesis tiap segmen validasi pada tiap evaluasi | Penelusuran segmen validasi yang memburuk |
| `latih/sesi.jsonl` | Perangkat, langkah awal dan akhir, durasi tiap sesi pelatihan | Analisis sumber daya |
| `latih/adapter_terbaik/` | Adapter checkpoint terbaik dan `terbaik.json` | Bagian 2 |
| `latih/run_meta_latih.json`, `latih/konsol.log` | Perangkat server, versi pustaka, keluaran terminal | Bab 6, Tabel spesifikasi server |
| `latih/kurva_per_epoch.png` | Loss dan WER validasi per epoch untuk pemantauan | Pemantauan |
| `uji_coba/latih/`, `uji_coba/kurva_per_epoch.png` | Uji coba 2 epoch: log, adapter, dan kurva (bukan hasil penelitian) | Pemeriksaan kode, perkiraan waktu |
| `_kerja/state/state_terakhir.pt` | State lengkap untuk melanjutkan (tetap di server, tidak di-commit) | Melanjutkan pelatihan |